# CH 담당 에이전트 고도화: A1 거시 · A2 재무 · A4 투자심리

`solar_pro_4.ipynb`(참고용)에서 CH 담당 세 관점만 떼어 고도화하는 작업 노트북입니다.

| 에이전트 | 관점 | 이번 고도화 |
| --- | --- | --- |
| A1 | 거시경제·변수 민감도 | 3개월·12개월 변화, 반도체 수출 증감, 환율 동조성(통계) |
| A2 | 재무·밸류에이션·건전성 | 분기 추세·전년 대비, PER 기준(TTM/연간) 분리, 이자보상배율 대용치 |
| A4 | 투자심리·수급 | 5·20·60일 수급 비교, 외국인 지분율 변화, 감성 추이, 무자료 표시 |

## 이 노트북의 핵심 규칙

1. **숫자는 코드가 만든다.** SQL·계산 결과를 `M1`, `F3` 같은 키로 등록합니다.
2. **LLM은 키만 쓴다.** 문장에 `{M1}`처럼 키를 넣으면, 코드가 실제 값·단위·기준일로 바꿉니다.
3. **없는 데이터는 '무자료'로 남긴다.** 0으로 채우거나 추정하지 않습니다.
4. **출력 형식은 팀 공통 계약을 따른다.** 오케스트레이터가 그대로 받아 쓸 수 있게 합니다.

**필요 패키지**: `psycopg2-binary`, `python-dotenv`, `langchain-upstage`, `langchain-core`

**필요 환경변수(.env)**: `UPSTAGE_API_KEY`, `DATABASE_URL` (선택: `UPSTAGE_MODEL_NAME`)

In [ ]:
# %pip install psycopg2-binary python-dotenv langchain-upstage langchain-core

## 0. 환경 설정

In [ ]:
import json
import math
import os
import re
from datetime import date, datetime
from decimal import Decimal
from typing import Any, Dict, List, Optional

import psycopg2
from dotenv import load_dotenv
from psycopg2.extras import RealDictCursor

load_dotenv()

UPSTAGE_MODEL_NAME = os.getenv("UPSTAGE_MODEL_NAME", "solar-pro4")
DATABASE_URL = os.getenv("DATABASE_URL", "")
TICKER = "005930"          # 분석 종목
MODE = "expert"            # expert | guide


def get_llm(temperature: float = 0.1):
    from langchain_upstage import ChatUpstage
    return ChatUpstage(model=UPSTAGE_MODEL_NAME, temperature=temperature)


def q(sql: str, params: tuple = ()) -> List[Dict[str, Any]]:
    """읽기 전용 조회. Decimal은 float으로 바꿔 돌려줍니다."""
    if not DATABASE_URL:
        raise RuntimeError("DATABASE_URL이 비어 있습니다. .env를 확인하세요.")
    with psycopg2.connect(DATABASE_URL) as conn, conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(sql, params)
        rows = cur.fetchall()
    return [{k: float(v) if isinstance(v, Decimal) else v for k, v in r.items()} for r in rows]


def company(ticker: str) -> Dict[str, Any]:
    rows = q("SELECT company_id, ticker, name, sector FROM core.company WHERE market = 'KR' AND ticker = %s", (ticker,))
    if not rows:
        raise ValueError(f"core.company에 KR {ticker}가 없습니다.")
    return rows[0]

## 1. 숫자 장부(Facts)와 공통 출력 계약

`Facts`는 에이전트가 인용할 수 있는 숫자의 목록입니다. 값마다 단위·기준일·출처·구분(실측/계산/대용치)을 붙입니다.
데이터가 없으면 `gap()`으로 '무자료' 사유를 남깁니다.

에이전트 결과(오케스트레이터에 넘기는 dict)의 키는 다음과 같습니다. 이전 노트북의 `analysis`, `source` 키도 유지합니다.

| 키 | 내용 |
| --- | --- |
| `agent_id`, `agent_name`, `status` | 번호·이름·상태(완료/실패/형식 오류) |
| `period` | 관측 기간과 자료 기준일 |
| `result` | LLM 구조화 출력(핵심진단·주장·긍정·리스크·미확인·다음점검·점수) |
| `facts`, `gaps` | 숫자 장부와 무자료 목록 |
| `checks` | 코드 검사 결과(없는 키 인용, 키 밖 숫자, 금지 표현) |
| `analysis` | 사람이 읽는 마크다운(키를 실제 값으로 바꾼 것) |

In [ ]:
KIND_LABEL = {"obs": "실측", "calc": "계산", "proxy": "대용치"}


def fmt(value: Optional[float], unit: str) -> str:
    """단위별 표기. 값 자체는 바꾸지 않고 보기만 정리합니다."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "무자료"
    v = float(value)
    if unit == "%":
        return f"{v:,.2f}%"
    if unit == "%p":
        return f"{v:+.2f}%p"
    if unit == "%chg":
        return f"{v:+.2f}%"
    if unit == "배":
        return f"{v:,.2f}배"
    if unit in ("KRW", "KRW±"):
        sign = "+" if unit == "KRW±" and v > 0 else ""
        if abs(v) >= 1e12:
            return f"{sign}{v / 1e12:,.2f}조원"
        if abs(v) >= 1e8:
            return f"{sign}{v / 1e8:,.0f}억원"
        return f"{sign}{v:,.2f}원"
    if unit == "USD":
        return f"{v / 1e8:,.2f}억달러" if abs(v) >= 1e8 else f"{v:,.0f}달러"
    if unit == "주":
        return f"{v:+,.0f}주"
    if unit == "score":
        return f"{v:+.3f}"
    if unit == "coef":
        return f"{v:+.2f}"
    return f"{v:,.4g}{unit}"


class Facts:
    def __init__(self, prefix: str):
        self.prefix = prefix
        self.items: Dict[str, Dict[str, Any]] = {}
        self.gaps: List[Dict[str, str]] = []

    def add(self, label: str, value: Any, unit: str, as_of: Any = None, source: str = "",
            kind: str = "obs", note: str = "") -> Optional[str]:
        if value is None:
            self.gap(label, "DB 값 없음")
            return None
        key = f"{self.prefix}{len(self.items) + 1}"
        self.items[key] = {
            "label": label, "value": float(value), "unit": unit, "text": fmt(value, unit),
            "as_of": str(as_of) if as_of is not None else None, "source": source,
            "kind": KIND_LABEL.get(kind, kind), "note": note,
        }
        return key

    def gap(self, label: str, reason: str) -> None:
        self.gaps.append({"label": label, "reason": reason})

    def for_prompt(self) -> str:
        lines = [f"- {k}: {v['label']} = {v['text']} (기준 {v['as_of']}, {v['kind']}, 출처 {v['source']})"
                 + (f" ※{v['note']}" if v["note"] else "") for k, v in self.items.items()]
        lines += [f"- 무자료: {g['label']} ({g['reason']})" for g in self.gaps]
        return "\n".join(lines)

    def table_md(self) -> str:
        if not self.items:
            return "_인용 가능한 숫자가 없습니다._"
        rows = ["| 키 | 항목 | 값 | 기준일 | 구분 | 출처 |", "| --- | --- | --- | --- | --- | --- |"]
        rows += [f"| {k} | {v['label']} | {v['text']} | {v['as_of'] or '-'} | {v['kind']} | {v['source']} |"
                 for k, v in self.items.items()]
        return "\n".join(rows)


def pct_change(new: Optional[float], old: Optional[float]) -> Optional[float]:
    if new is None or old in (None, 0):
        return None
    return (new / old - 1) * 100


def corr_beta(xs: List[float], ys: List[float]):
    """상관계수와 베타(y를 x로 단순회귀한 기울기). 표본이 적으면 None."""
    n = len(xs)
    if n < 30:
        return None, None, n
    mx, my = sum(xs) / n, sum(ys) / n
    sxx = sum((x - mx) ** 2 for x in xs)
    syy = sum((y - my) ** 2 for y in ys)
    sxy = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    if sxx == 0 or syy == 0:
        return None, None, n
    return sxy / math.sqrt(sxx * syy), sxy / sxx, n

## 2. A1 거시경제·변수 민감도 데이터

- 지표별 최신값과 3개월·12개월 전 대비 변화 (금리는 %p, 나머지는 %)
- 반도체 수출(관세청): 최근 3개월 합계의 전년 같은 기간 대비 증감
- **환율 동조성**: 최근 250거래일 종목 일간 수익률과 원/달러 일간 변화의 상관·베타
  - 이것은 통계적 동조성이며 기업의 환율 민감도(노출)가 아닙니다. 노출을 계산할 수출 비중·외화 매출 데이터는 DB에 없습니다.
- 관련 뉴스 제목(정성 근거, `D` 키)

In [ ]:
MACRO_SERIES = {
    # series_id: (표시 이름, 단위, 변화 방식)  변화 방식: pp=%p 차이, pct=증감률
    "ECOS_BASE_RATE": ("한국 기준금리", "%", "pp"),
    "FEDFUNDS": ("미국 기준금리(실효)", "%", "pp"),
    "DGS10": ("미국 10년 국채금리", "%", "pp"),
    "ECOS_USDKRW": ("원/달러 환율", "KRW", "pct"),
    "DEXUSEU": ("달러/유로 환율", "", "pct"),
    "DEXTAUS": ("대만달러/달러 환율", "", "pct"),
    "IPG3344S": ("미국 반도체·전자부품 산업생산지수", "", "pct"),
    "PCU33443344": ("미국 반도체·전자부품 생산자물가지수", "", "pct"),
}
EXPORT_SERIES = {
    "KCS_EXP_854232_USD": "반도체 메모리 수출(최근 3개월 합계)",
    "KCS_EXP_854231_USD": "반도체 프로세서·컨트롤러 수출(최근 3개월 합계)",
}

SQL_MACRO_CHANGES = """
    WITH last AS (
        SELECT DISTINCT ON (series_id) series_id, obs_date, value
        FROM core.macro_obs
        WHERE series_id = ANY(%s)
        ORDER BY series_id, obs_date DESC
    )
    SELECT l.series_id, l.obs_date::text AS obs_date, l.value,
        (SELECT o.value FROM core.macro_obs o WHERE o.series_id = l.series_id
           AND o.obs_date <= l.obs_date - INTERVAL '3 months' ORDER BY o.obs_date DESC LIMIT 1) AS v_3m,
        (SELECT o.value FROM core.macro_obs o WHERE o.series_id = l.series_id
           AND o.obs_date <= l.obs_date - INTERVAL '12 months' ORDER BY o.obs_date DESC LIMIT 1) AS v_12m
    FROM last l;
"""
SQL_CPI_YOY = """
    SELECT obs_date::text AS obs_date, value,
        (SELECT o.value FROM core.macro_obs o WHERE o.series_id = 'CPIAUCSL'
           AND o.obs_date <= m.obs_date - INTERVAL '12 months' ORDER BY o.obs_date DESC LIMIT 1) AS v_12m
    FROM core.macro_obs m WHERE series_id = 'CPIAUCSL' ORDER BY obs_date DESC LIMIT 1;
"""
SQL_EXPORTS = """
    SELECT series_id, obs_date, value FROM core.macro_obs
    WHERE series_id = ANY(%s) AND obs_date >= (SELECT max(obs_date) FROM core.macro_obs WHERE series_id = ANY(%s)) - INTERVAL '15 months'
    ORDER BY series_id, obs_date;
"""
SQL_FX_COMOVE = """
    WITH px AS (
        SELECT trade_date, close, LAG(close) OVER (ORDER BY trade_date) AS prev
        FROM core.price_daily WHERE company_id = %s
    ), fx AS (
        SELECT obs_date, value, LAG(value) OVER (ORDER BY obs_date) AS prev
        FROM core.macro_obs WHERE series_id = 'ECOS_USDKRW'
    )
    SELECT px.trade_date::text AS d, px.close / px.prev - 1 AS r_stock, fx.value / fx.prev - 1 AS r_fx
    FROM px JOIN fx ON fx.obs_date = px.trade_date
    WHERE px.prev IS NOT NULL AND fx.prev IS NOT NULL
    ORDER BY px.trade_date DESC LIMIT 250;
"""
SQL_DOCS = """
    SELECT d.source, d.title, d.published_at::date::text AS published
    FROM rag.document d
    WHERE d.company_id = %s AND d.source = 'naver_news' AND d.title ~ %s
    ORDER BY d.published_at DESC NULLS LAST LIMIT %s;
"""


def fetch_macro(co: Dict[str, Any]) -> Dict[str, Any]:
    f = Facts("M")
    src = "core.macro_obs (ECOS/FRED/관세청)"
    rows = {r["series_id"]: r for r in q(SQL_MACRO_CHANGES, (list(MACRO_SERIES),))}
    for sid, (name, unit, how) in MACRO_SERIES.items():
        r = rows.get(sid)
        if not r:
            f.gap(name, "시계열 없음")
            continue
        f.add(name, r["value"], unit, r["obs_date"], f"{src} {sid}")
        for label, old in (("3개월", r["v_3m"]), ("12개월", r["v_12m"])):
            if how == "pp":
                chg = None if old is None else r["value"] - old
                f.add(f"{name} {label} 변화", chg, "%p", r["obs_date"], f"{src} {sid}", kind="calc")
            else:
                f.add(f"{name} {label} 변화율", pct_change(r["value"], old), "%chg", r["obs_date"], f"{src} {sid}", kind="calc")

    cpi = q(SQL_CPI_YOY)
    if cpi:
        f.add("미국 CPI 전년 대비", pct_change(cpi[0]["value"], cpi[0]["v_12m"]), "%chg", cpi[0]["obs_date"],
              f"{src} CPIAUCSL", kind="calc")
    f.gap("한국 소비자물가·국고채 금리", "ECOS 해당 시계열 미수집")

    exp_rows = q(SQL_EXPORTS, (list(EXPORT_SERIES), list(EXPORT_SERIES)))
    for sid, label in EXPORT_SERIES.items():
        vals = [(r["obs_date"], r["value"]) for r in exp_rows if r["series_id"] == sid]
        if len(vals) < 15:
            f.gap(label, "15개월 미만 관측")
            continue
        recent = sum(v for _, v in vals[-3:])
        year_ago = sum(v for _, v in vals[-15:-12])
        f.add(label, recent, "USD", vals[-1][0], f"관세청 {sid}")
        f.add(f"{label} 전년 동기 대비", pct_change(recent, year_ago), "%chg", vals[-1][0], f"관세청 {sid}", kind="calc")

    pairs = q(SQL_FX_COMOVE, (co["company_id"],))
    rho, beta, n = corr_beta([p["r_fx"] for p in pairs], [p["r_stock"] for p in pairs])
    period = f"{pairs[-1]['d']} ~ {pairs[0]['d']}" if pairs else "-"
    note = f"{n}거래일 일간 변화 기준. 통계적 동조성이며 인과·노출이 아님"
    f.add("주가-원/달러 상관계수", rho, "coef", period, "core.price_daily × ECOS_USDKRW", kind="calc", note=note)
    f.add("주가-원/달러 베타", beta, "coef", period, "core.price_daily × ECOS_USDKRW", kind="calc",
          note="원/달러 1% 변화당 주가 평균 변화(%)")
    f.gap("기업 환율 노출(수출 비중·외화 매출·헤지)", "DART 사업보고서 본문 미수집")

    docs = q(SQL_DOCS, (co["company_id"], "환율|금리|수출|관세", 5))
    return {"facts": f, "docs": docs, "period": "거시지표 최근 12개월, 동조성 최근 250거래일"}

## 3. A2 재무·밸류에이션·건전성 데이터

- TTM 실적·멀티플 (`core.v_ttm`, 종목당 1행)
- 최근 분기 매출·영업이익의 전년 같은 분기 대비 증감 (`core.financial_quarter`)
- **PER 기준 분리**: TTM PER과 연간(FY) PER을 다른 키로 등록해 섞이지 않게 합니다
- Peer 중앙값 대비 차이 (`core.feature_snapshot`)
- **이자보상배율 대용치**: 연간 영업이익 ÷ 현금흐름표 '이자의 지급'. 손익계산서 이자비용이 아니므로 대용치로 표시합니다

In [ ]:
SQL_TTM = """
    SELECT ttm_end::text AS ttm_end, bs_end::text AS bs_end, revenue_ttm, operating_income_ttm, net_income_ttm,
           op_margin_ttm, roe_ttm, debt_ratio, per_ttm, pbr, equity, currency
    FROM core.v_ttm WHERE market = 'KR' AND ticker = %s;
"""
SQL_QUARTERS = """
    SELECT period_end::text AS period_end, fiscal_year, fiscal_q,
           max(amount) FILTER (WHERE metric = 'revenue') AS revenue,
           max(amount) FILTER (WHERE metric = 'operating_income') AS op_income
    FROM core.financial_quarter
    WHERE company_id = %s AND metric IN ('revenue', 'operating_income')
    GROUP BY period_end, fiscal_year, fiscal_q
    ORDER BY period_end DESC LIMIT 8;
"""
SQL_VALUATION = """
    SELECT fiscal_year, price_date::text AS price_date, per, pbr
    FROM core.v_valuation WHERE market = 'KR' AND ticker = %s
    ORDER BY fiscal_year DESC LIMIT 1;
"""
SQL_FEATURES = """
    SELECT feature, value, unit, note, as_of::text AS as_of
    FROM core.feature_snapshot
    WHERE company_id = %s AND perspective = ANY(%s)
      AND as_of = (SELECT max(as_of) FROM core.feature_snapshot WHERE company_id = %s);
"""
SQL_INTEREST = """
    SELECT a.fiscal_year, a.operating_income, f.amount AS interest_paid
    FROM core.v_fundamentals_annual a
    JOIN core.financial_fact f ON f.company_id = a.company_id AND f.fiscal_year = a.fiscal_year
    WHERE a.company_id = %s AND f.account_nm = '이자의 지급' AND f.report_code = '11011' AND f.fs_div = 'CFS'
    ORDER BY a.fiscal_year DESC LIMIT 1;
"""
PEER_FEATURES = {
    "op_margin_vs_peer_median": ("영업이익률 Peer 중앙값 대비", "%p", 100),
    "roe_vs_peer_median": ("ROE Peer 중앙값 대비", "%p", 100),
    "per_ttm_vs_peer_median": ("PER(TTM) Peer 중앙값 대비", "%chg", 100),
    "pbr_vs_peer_median": ("PBR Peer 중앙값 대비", "%chg", 100),
}


def fetch_financial(co: Dict[str, Any]) -> Dict[str, Any]:
    f = Facts("F")
    ttm = q(SQL_TTM, (co["ticker"],))
    if not ttm:
        f.gap("TTM 실적·멀티플", "core.v_ttm에 이 종목 없음 (재무는 일부 종목만 수집됨)")
    else:
        t = ttm[0]
        s = "core.v_ttm (OpenDART TTM)"
        f.add("매출(TTM)", t["revenue_ttm"], "KRW", t["ttm_end"], s)
        f.add("영업이익(TTM)", t["operating_income_ttm"], "KRW", t["ttm_end"], s)
        f.add("순이익(TTM)", t["net_income_ttm"], "KRW", t["ttm_end"], s)
        f.add("영업이익률(TTM)", None if t["op_margin_ttm"] is None else t["op_margin_ttm"] * 100, "%", t["ttm_end"], s, kind="calc")
        f.add("ROE(TTM)", None if t["roe_ttm"] is None else t["roe_ttm"] * 100, "%", t["ttm_end"], s, kind="calc")
        f.add("부채비율", None if t["debt_ratio"] is None else t["debt_ratio"] * 100, "%", t["bs_end"], s, kind="calc")
        f.add("PER(TTM 기준)", t["per_ttm"], "배", t["ttm_end"], s, kind="calc", note="최근 4개 분기 순이익 기준")
        f.add("PBR", t["pbr"], "배", t["bs_end"], s, kind="calc")
        if t["equity"] is not None and t["equity"] <= 0:
            f.add("자본총계(완전자본잠식)", t["equity"], "KRW", t["bs_end"], s)

    qs = q(SQL_QUARTERS, (co["company_id"],))
    if len(qs) >= 5:
        now, prev = qs[0], qs[4]
        lab = f"{now['fiscal_year']}년 {now['fiscal_q']}분기"
        s = "core.financial_quarter (분기 단독값)"
        f.add(f"{lab} 매출", now["revenue"], "KRW", now["period_end"], s)
        f.add(f"{lab} 매출 전년 동기 대비", pct_change(now["revenue"], prev["revenue"]), "%chg", now["period_end"], s, kind="calc")
        f.add(f"{lab} 영업이익", now["op_income"], "KRW", now["period_end"], s)
        f.add(f"{lab} 영업이익 전년 동기 대비", pct_change(now["op_income"], prev["op_income"]), "%chg", now["period_end"], s,
              kind="calc", note="전년 영업이익이 음수면 해석 주의")
        if now["revenue"] and now["op_income"] is not None and prev["revenue"] and prev["op_income"] is not None:
            f.add(f"{lab} 영업이익률 전년 동기 대비 변화",
                  (now["op_income"] / now["revenue"] - prev["op_income"] / prev["revenue"]) * 100, "%p",
                  now["period_end"], s, kind="calc")
    else:
        f.gap("분기 추세", "분기 단독값 5개 미만")

    val = q(SQL_VALUATION, (co["ticker"],))
    if val:
        v = val[0]
        f.add(f"PER(연간 FY{v['fiscal_year']} 기준)", v["per"], "배", v["price_date"], "core.v_valuation", kind="calc",
              note=f"주가 {v['price_date']} ÷ FY{v['fiscal_year']} 주당순이익. TTM PER과 다른 기준")

    feats = {r["feature"]: r for r in q(SQL_FEATURES, (co["company_id"], ["valuation", "fundamentals", "risk"], co["company_id"]))}
    for feat, (label, unit, scale) in PEER_FEATURES.items():
        r = feats.get(feat)
        f.add(label, None if not r or r["value"] is None else r["value"] * scale, unit, r and r["as_of"],
              "core.feature_snapshot", kind="calc", note="Peer 중앙값 대비 차이")
    for feat, label in (("max_drawdown_250d", "최근 250일 최대 낙폭"), ("volatility_60d", "60일 변동성")):
        r = feats.get(feat)
        f.add(label, None if not r or r["value"] is None else r["value"] * 100, "%", r and r["as_of"],
              "core.feature_snapshot", kind="calc")

    ir = q(SQL_INTEREST, (co["company_id"],))
    if ir and ir[0]["interest_paid"]:
        r = ir[0]
        f.add(f"이자보상배율 대용치(FY{r['fiscal_year']})", r["operating_income"] / r["interest_paid"], "배",
              f"FY{r['fiscal_year']}", "v_fundamentals_annual ÷ financial_fact(이자의 지급)", kind="proxy",
              note="영업이익 ÷ 현금흐름표 이자 지급액. 손익계산서 이자비용 기준이 아님")
    else:
        f.gap("이자보상배율", "사업보고서 '이자의 지급' 계정 없음")
    f.gap("EV/EBITDA", "감가상각비·순차입금 미수집")
    f.gap("감사의견", "미수집")
    return {"facts": f, "docs": [], "period": "TTM·최근 분기와 전년 동기, 연간 FY"}

## 4. A4 투자심리·수급 데이터

- 외국인·기관·개인 순매수: 5·20·60거래일 누적 (수량, 종가 기준 금액 근사)
- 외국인 지분율: 최신값과 20·60거래일 전 대비 변화
- 감성·언급량·검색 관심도: `feature_snapshot`의 계산값 (키워드 사전 기반이며 학습 모델이 아님)
- 주식대차(공매도 대용): 테이블이 비어 있으면 '무자료'
- **중복 근거 경고**: 뉴스 감성·언급량·검색 관심도는 같은 사건을 반영할 수 있어 독립 근거로 세지 않게 프롬프트에서 막습니다

In [ ]:
SQL_FLOWS = """
    SELECT f.trade_date::text AS d, f.foreign_net_qty, f.institution_net_qty, f.individual_net_qty,
           f.foreign_hold_pct, p.close
    FROM core.investor_flow_daily f
    LEFT JOIN core.price_daily p ON p.company_id = f.company_id AND p.trade_date = f.trade_date
    WHERE f.company_id = %s
    ORDER BY f.trade_date DESC LIMIT 60;
"""
SQL_LENDING = """
    SELECT trade_date::text AS d, balance_qty FROM core.stock_lending_daily
    WHERE company_id = %s ORDER BY trade_date DESC LIMIT 20;
"""
SENTIMENT_FEATURES = {
    "news_sentiment_7d": ("뉴스 감성(7일 가중)", "score"),
    "crowd_sentiment_7d": ("블로그·카페 감성(7일 가중)", "score"),
    "sentiment_1d_vs_7d": ("당일 감성 − 7일 감성", "score"),
    "crowd_sentiment_chg": ("투자자 감성 변화(최근 7일 − 직전 8~30일)", "score"),
    "search_attention_7d_vs_90d": ("검색 관심도(최근 7일 ÷ 직전 90일)", "배"),
    "mentions_1d_vs_30d": ("언급량(당일 ÷ 이전 일평균)", "배"),
}


def fetch_sentiment(co: Dict[str, Any]) -> Dict[str, Any]:
    f = Facts("S")
    rows = q(SQL_FLOWS, (co["company_id"],))
    s = "core.investor_flow_daily (네이버 투자자별 매매)"
    period = "-"
    if not rows:
        f.gap("투자자별 수급", "수집 데이터 없음")
    else:
        period = f"{rows[-1]['d']} ~ {rows[0]['d']} ({len(rows)}거래일)"
        for who, col in (("외국인", "foreign_net_qty"), ("기관", "institution_net_qty"), ("개인", "individual_net_qty")):
            for n in (5, 20, 60):
                if len(rows) < n:
                    continue
                win = rows[:n]
                f.add(f"{who} {n}일 순매수 수량", sum(r[col] or 0 for r in win), "주", rows[0]["d"], s, kind="calc")
                if n == 20 and all(r["close"] is not None for r in win):
                    f.add(f"{who} 20일 순매수 금액(근사)", sum((r[col] or 0) * r["close"] for r in win), "KRW±", rows[0]["d"],
                          s + " × 종가", kind="proxy", note="일별 순매수 수량 × 종가. 실제 체결 금액과 다름")
        f.add("외국인 지분율", rows[0]["foreign_hold_pct"], "%", rows[0]["d"], s)
        for n in (20, 60):
            if len(rows) >= n and rows[n - 1]["foreign_hold_pct"] is not None:
                f.add(f"외국인 지분율 {n}일 변화", rows[0]["foreign_hold_pct"] - rows[n - 1]["foreign_hold_pct"], "%p",
                      rows[0]["d"], s, kind="calc")

    if not q(SQL_LENDING, (co["company_id"],)):
        f.gap("주식대차 잔고(공매도 대용)", "core.stock_lending_daily 비어 있음")

    feats = {r["feature"]: r for r in q(SQL_FEATURES, (co["company_id"], ["supply_demand"], co["company_id"]))}
    for feat, (label, unit) in SENTIMENT_FEATURES.items():
        r = feats.get(feat)
        if r and r["value"] is not None:
            f.add(label, r["value"], unit, r["as_of"], "core.feature_snapshot", kind="calc",
                  note=(r["note"] or "")[:60] + " / 키워드 사전 기반")
        else:
            f.gap(label, (r and r["note"]) or "값 없음 (누적 일수 부족 등)")
    for feat, label in (("return_20d", "20일 수익률"), ("volume_ratio_20_60", "거래량(20일 ÷ 60일)")):
        r = feats.get(feat)
        unit = "%" if feat.startswith("return") else "배"
        val = None if not r or r["value"] is None else r["value"] * (100 if unit == "%" else 1)
        f.add(label, val, unit, r and r["as_of"], "core.feature_snapshot", kind="calc")
    return {"facts": f, "docs": [], "period": period}

## 5. 에이전트 실행: 프롬프트 → 구조화 출력 → 코드 검사 → 렌더링

LLM은 JSON만 돌려주고, 숫자는 `{키}`로만 씁니다. 코드가 다음을 검사합니다.

- 장부에 없는 키를 인용했는가
- 키 밖에 단위 붙은 숫자를 직접 썼는가 (환각 의심)
- '사실' 주장에 근거 키가 없는가
- 매수·매도 추천, 목표주가 같은 금지 표현이 있는가

In [ ]:
COMMON_RULES = """[절대 준수 원칙]
1. 숫자를 직접 쓰지 마십시오. 숫자가 필요하면 [숫자 장부]의 키를 중괄호로 쓰십시오. 예: 원/달러 환율은 {M7}입니다.
2. 장부에 없는 키나 '무자료' 항목의 값을 만들지 마십시오. 무자료 항목은 '미확인'에 적으십시오.
3. '매수 추천', '매도 추천', '목표주가' 표현은 금지합니다.
4. 각 주장은 사실(데이터가 직접 보여줌)·해석(원인 설명)·예측(앞으로) 중 하나로 구분하십시오.
5. 통계적 상관을 인과나 기업 노출로 표현하지 마십시오."""

OUTPUT_SCHEMA = """반드시 아래 JSON만 출력하십시오 (코드블록 없이):
{
  "핵심진단": "1~2문장",
  "주장": [{"문장": "...", "구분": "사실|해석|예측", "근거": ["키", "..."]}],
  "긍정요인": ["..."],
  "리스크요인": ["..."],
  "미확인": ["판단에 필요하지만 데이터가 없는 것"],
  "다음점검": ["다음에 확인할 지표와 시점"],
  "자체점수": 1~10 정수,
  "점수이유": "..."
}"""

AGENTS = {
    1: {
        "name": "거시경제 및 변수 민감도 분석 에이전트 (Macro & Sensitivity)",
        "fetch": fetch_macro,
        "role": ("금리·환율·물가·반도체 수출 등 거시변수의 최근 변화와, 그 변화가 기업 매출·비용·수요로 전달되는 경로를 분석합니다. "
                 "환율 동조성 수치는 통계적 관계일 뿐 기업의 환율 노출이 아님을 분명히 하십시오."),
    },
    2: {
        "name": "재무·밸류에이션 및 건전성 분석 에이전트 (Financial & Risk)",
        "fetch": fetch_financial,
        "role": ("TTM 실적과 최근 분기 추세, 밸류에이션 멀티플, Peer 대비 위치, 부채·이자 감당력을 심사합니다. "
                 "PER은 TTM 기준과 연간 기준을 반드시 구분해 쓰고, 대용치 지표는 대용치라고 밝히십시오."),
    },
    4: {
        "name": "투자심리 및 수급 분석 에이전트 (Sentiment & Supply/Demand)",
        "fetch": fetch_sentiment,
        "role": ("외국인·기관·개인 수급의 기간별 흐름, 외국인 지분율 변화, 뉴스·커뮤니티 감성과 검색 관심도를 분석합니다. "
                 "감성·언급량·검색 관심도는 같은 사건을 반영할 수 있으므로 독립된 여러 근거로 세지 마십시오. "
                 "순매수 기록만으로 매수 의도나 정보 보유를 추정하지 마십시오."),
    },
}

MODE_GUIDE = {
    "expert": "전문가 모드: 핵심 지표와 인과 경로 위주로 간결하게 쓰십시오.",
    "guide": "입문자 모드: 전문 용어가 처음 나오면 괄호로 쉬운 풀이와 '왜 보는지'를 덧붙이십시오.",
}

PLACEHOLDER = re.compile(r"\{([A-Z]\d+)\}")
LOOSE_NUMBER = re.compile(r"\d[\d,]*(?:\.\d+)?\s*(?:%p|%|원|배|조|억|달러|bp)")
BANNED = ("매수 추천", "매도 추천", "목표주가")


def parse_json(text: str) -> Optional[Dict[str, Any]]:
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def check_output(result: Dict[str, Any], facts: Facts) -> Dict[str, List[str]]:
    blob = json.dumps(result, ensure_ascii=False)
    cited = set(PLACEHOLDER.findall(blob))
    for claim in result.get("주장", []):
        cited.update(k for k in claim.get("근거", []) if re.fullmatch(r"[A-Z]\d+", str(k)))
    stripped = PLACEHOLDER.sub("", blob)
    return {
        "없는 키 인용": sorted(k for k in cited if k not in facts.items),
        "키 밖 숫자": sorted(set(LOOSE_NUMBER.findall(stripped))),
        "근거 없는 사실 주장": [c.get("문장", "") for c in result.get("주장", []) if c.get("구분") == "사실" and not c.get("근거")],
        "금지 표현": [w for w in BANNED if w in blob],
    }


def render(text: str, facts: Facts) -> str:
    def sub(m):
        item = facts.items.get(m.group(1))
        return f"**{item['text']}**`[{m.group(1)}]`" if item else f"⟨없는 키 {m.group(1)}⟩"
    return PLACEHOLDER.sub(sub, str(text))


def to_markdown(agent_id: int, name: str, period: str, result: Dict[str, Any], facts: Facts,
                checks: Dict[str, List[str]], docs: List[Dict[str, Any]]) -> str:
    r = lambda t: render(t, facts)  # noqa: E731
    out = [f"- **관측 기간**: {period}", f"- **핵심 진단**: {r(result.get('핵심진단', ''))}", "", "**주장과 근거**", ""]
    for c in result.get("주장", []):
        keys = ", ".join(c.get("근거", [])) or "근거 없음"
        out.append(f"- [{c.get('구분', '?')}] {r(c.get('문장', ''))} (근거: {keys})")
    for title, key in (("긍정 요인", "긍정요인"), ("리스크 요인", "리스크요인"), ("미확인", "미확인"), ("다음 점검", "다음점검")):
        items = result.get(key) or []
        out += ["", f"**{title}**", ""] + [f"- {r(x)}" for x in items] if items else []
    out += ["", f"**자체 점수**: {result.get('자체점수', '-')} / 10 · {r(result.get('점수이유', ''))}", "",
            "**숫자 장부**", "", facts.table_md()]
    if facts.gaps:
        out += ["", "**무자료**", ""] + [f"- {g['label']}: {g['reason']}" for g in facts.gaps]
    if docs:
        out += ["", "**참고 뉴스(제목)**", ""] + [f"- {d['published']} · {d['title']}" for d in docs]
    problems = {k: v for k, v in checks.items() if v}
    out += ["", "**코드 검사**: " + ("통과" if not problems else "; ".join(f"{k} {v}" for k, v in problems.items()))]
    return "\n".join(out)


def run_agent(agent_id: int, ticker: str = TICKER, mode: str = MODE, llm=None, data: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    spec = AGENTS[agent_id]
    co = company(ticker)
    data = data or spec["fetch"](co)
    facts: Facts = data["facts"]
    docs_txt = "\n".join(f"- D{i + 1}: {d['published']} {d['title']}" for i, d in enumerate(data["docs"])) or "- (없음)"
    system = f"""당신은 OpenResearch의 [A{agent_id}: {spec['name']}]입니다.
{spec['role']}

{COMMON_RULES}

[출력 모드] {MODE_GUIDE[mode]}

{OUTPUT_SCHEMA}"""
    user = f"""[대상] {co['name']} ({co['ticker']})
[관측 기간] {data['period']}

[숫자 장부] (값을 쓰려면 키를 중괄호로)
{facts.for_prompt()}

[참고 뉴스 제목] (정성 근거, 근거 키로 D번호 사용 가능)
{docs_txt}"""
    base = {"agent_id": agent_id, "agent_name": spec["name"], "period": data["period"],
            "facts": facts.items, "gaps": facts.gaps, "source": "Neon " + ", ".join(sorted({v["source"].split(" (")[0] for v in facts.items.values()}))}
    try:
        from langchain_core.messages import HumanMessage, SystemMessage
        raw = (llm or get_llm()).invoke([SystemMessage(content=system), HumanMessage(content=user)]).content
    except Exception as e:  # 실패도 상태로 남김
        return {**base, "status": "실패", "result": {}, "checks": {}, "analysis": f"(분석 실패: {type(e).__name__}: {e})"}
    result = parse_json(raw)
    if result is None:
        return {**base, "status": "형식 오류", "result": {}, "checks": {}, "raw": raw,
                "analysis": "(JSON 형식이 아니어서 렌더링하지 못했습니다. raw 키를 확인하세요.)\n\n" + raw}
    # D키(뉴스)는 장부 밖이므로 검사 전에 허용 목록에 넣음
    for i, d in enumerate(data["docs"]):
        facts.items.setdefault(f"D{i + 1}", {"label": d["title"], "value": None, "unit": "", "text": f"뉴스 {d['published']}",
                                             "as_of": d["published"], "source": "rag.document naver_news", "kind": "문서", "note": ""})
    checks = check_output(result, facts)
    md = to_markdown(agent_id, spec["name"], data["period"], result, facts, checks, data["docs"])
    status = "완료" if not any(checks.values()) else "완료(검사 경고)"
    return {**base, "status": status, "result": result, "checks": checks, "analysis": md}

## 6. 데이터만 먼저 확인 (LLM 호출 없음)

Solar를 부르기 전에 장부에 어떤 숫자가 들어가는지 확인합니다. 숫자가 이상하면 여기서 SQL을 고칩니다.

In [ ]:
from IPython.display import Markdown, display  # noqa: E402

co = company(TICKER)
DATA = {1: fetch_macro(co), 2: fetch_financial(co), 4: fetch_sentiment(co)}
for aid, d in DATA.items():
    md = f"### A{aid} 숫자 장부 · {d['period']}\n\n{d['facts'].table_md()}"
    if d["facts"].gaps:
        md += "\n\n**무자료**\n\n" + "\n".join(f"- {g['label']}: {g['reason']}" for g in d["facts"].gaps)
    display(Markdown(md))

## 7. 에이전트 하나씩 실행

위에서 확인한 데이터(`DATA`)를 그대로 넘겨 DB를 다시 조회하지 않습니다. 번호만 바꿔 실행하세요.

In [ ]:
res = run_agent(1, data=DATA[1])
print(res["status"])
display(Markdown(res["analysis"]))

In [ ]:
res = run_agent(2, data=DATA[2])
print(res["status"])
display(Markdown(res["analysis"]))

In [ ]:
res = run_agent(4, data=DATA[4])
print(res["status"])
display(Markdown(res["analysis"]))

## 8. 오케스트레이터에 넘길 묶음

세 결과를 리스트로 모읍니다. 키 구성은 위 1절의 표를 따릅니다. 오케스트레이터 담당 팀원에게는 이 형식을 공유하면 됩니다.

In [ ]:
MY_OUTPUTS = [run_agent(aid, data=DATA[aid]) for aid in (1, 2, 4)]
for o in MY_OUTPUTS:
    print(f"A{o['agent_id']} {o['status']} · 숫자 {len(o['facts'])}개 · 무자료 {len(o['gaps'])}개 · 경고 {sum(len(v) for v in o['checks'].values())}건")